# 1. Dados de inicialização

1.1 Dar mount no Drive e carregar o modelo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf
import numpy as np

# Caminho para o modelo
model_path = '/content/drive/My Drive/Projeto/PARA UPLOAD/TLSeg_128_6.keras'

try:
    # Carregar o modelo
    model = tf.keras.models.load_model(model_path)
    print("Modelo carregado com sucesso!")

    # Exibir o resumo da arquitetura
    model.summary()

except Exception as e:
    print(f"Erro ao carregar o modelo: {e}")

1.2 Acessar o sub-modelo

In [ ]:
# O EfficientNetB0 está no índice 1, logo após o Input
# O nome 'efficientnetb0' veio do summary
inner_model = model.get_layer('efficientnetb0')

# 2. Definir o nome da camada alvo interna do EfficientNet
# 'top_activation' é o padrão do EfficientNet para a última camada de features antes do pooling
target_layer_name = 'top_activation'

try:
    # Tentar pegar a camada para ver se ela existe mesmo
    target_layer = inner_model.get_layer(target_layer_name)
    print(f"Sucesso - Camada '{target_layer_name}' encontrada dentro do EfficientNet.")
    print(f"Shape de saída da camada: {target_layer.output_shape}")

except ValueError:
    print(f"Erro: A camada '{target_layer_name}' não foi encontrada.")
    print("Listando as últimas 10 camadas do modelo interno para conferir:")
    for layer in inner_model.layers[-10:]:
        print(layer.name)

# É normal dar erro logo após de pegar a camada

1.3 Criar o Feature Extractor

In [ ]:
# O inner_model é 'efficientnetb0' do summary
last_conv_model = tf.keras.Model(inputs=inner_model.inputs, outputs=target_layer.output)

# 2. Criar o modelo classificador (O resto do modelo após o EfficientNet)
# Pega todas as camadas que vêm DEPOIS do efficientnetb0 no modelo original
classifier_input = tf.keras.Input(shape=target_layer.output.shape[1:])
x = classifier_input

# Encontrar onde o efficientnetb0 está na lista de camadas
eff_layer_index = model.layers.index(inner_model)

# Iterar sobre as camadas seguintes para recriar o cabeçalho de classificação
for layer in model.layers[eff_layer_index + 1:]:
    x = layer(x)

classifier_model = tf.keras.Model(inputs=classifier_input, outputs=x)

print("--- Identificação concluída ---")
print(f"Extrator de Features: Entrada {last_conv_model.input_shape} -> Saída {last_conv_model.output_shape}")
print(f"Classificador: Entrada {classifier_model.input_shape} -> Saída {classifier_model.output_shape}")

1.4 Verificar o tipo de pré-processamento

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input

# 1. Carrega a imagem original (pixels 0 a 255)
img_path = '/content/drive/My Drive/Projeto/imagens_segmentadas/Glioma/segmented_G_125.jpg'
img = tf.keras.preprocessing.image.load_img(img_path, target_size=(128, 128))
img_array = tf.keras.preprocessing.image.img_to_array(img) # Vira array numpy
img_batch = np.expand_dims(img_array, axis=0) # (1, 128, 128, 3)

print("--- Teste de Sanidade ---")
print("Classes identificadas: ['Normal', 'glioma_tumor', 'meningioma_tumor', 'pituitary_tumor']")

# CENÁRIO A: Passar 0 a 255 direto
# (Muitos EfficientNets salvos .keras já têm o rescale embutido)
pred_a = model.predict(img_batch, verbose=0)
score_a = np.max(pred_a)
print(f"A) Entrada 0-255: Confiança {score_a:.4f} -> Classe {np.argmax(pred_a)}")

# CENÁRIO B: Normalizar entre 0 e 1
pred_b = model.predict(img_batch / 255.0, verbose=0)
score_b = np.max(pred_b)
print(f"B) Entrada 0-1:   Confiança {score_b:.4f} -> Classe {np.argmax(pred_b)}")

# CENÁRIO C: Usar preprocess_input do Keras
img_preprocessed = preprocess_input(img_batch.copy())
pred_c = model.predict(img_preprocessed, verbose=0)
score_c = np.max(pred_c)
print(f"C) Preprocess Keras: Confiança {score_c:.4f} -> Classe {np.argmax(pred_c)}")

# 2. Módulos únicos - GINI | AOPC

In [ ]:
import numpy as np
import cv2

# --- 1. ÍNDICE GINI ---
def calculate_gini_index(heatmap: np.ndarray) -> float:
    flat_weights = np.abs(heatmap.flatten())
    if np.sum(flat_weights) == 0:
        return 0.0
    sorted_weights = np.sort(flat_weights)
    n = len(sorted_weights)
    index = np.arange(1, n + 1)
    gini = np.sum((2 * index - n - 1) * sorted_weights) / (n * np.sum(sorted_weights))
    return float(gini)


# --- 2. AOPC ---
def calculate_aopc(model, img_array: np.ndarray, heatmap: np.ndarray, k_steps: int = 10, mask_value: float = 0.0) -> float:
    if len(img_array.shape) == 3:
        img_batch = np.expand_dims(img_array, axis=0)
    else:
        img_batch = img_array.copy()
        img_array = img_batch[0]

    orig_preds = model.predict(img_batch, verbose=0)
    target_class = np.argmax(orig_preds[0])
    p_original = orig_preds[0, target_class]

    flat_indices = np.argsort(heatmap.flatten())[::-1]
    pixels_per_step = len(flat_indices) // k_steps

    probability_drops = []
    perturbed_img = img_array.copy()

    for step in range(1, k_steps + 1):
        current_mask_indices = flat_indices[: step * pixels_per_step]
        rows, cols = np.unravel_index(current_mask_indices, heatmap.shape[:2])

        perturbed_img[rows, cols, :] = mask_value

        new_batch = np.expand_dims(perturbed_img, axis=0)
        new_preds = model.predict(new_batch, verbose=0)
        p_perturbed = new_preds[0, target_class]

        probability_drops.append(p_original - p_perturbed)

    return float(np.mean(probability_drops))


# --- 3. ADAPTADOR/PADRONIZADOR DE HEATMAPS ---
def extract_heatmap_2d(xai_type: str, raw_output) -> np.ndarray:
    """Padroniza a saída do Grad-CAM++, LIME e SHAP para uma matriz 2D (128x128)."""
    if xai_type == "GradCAM++":
        heatmap = cv2.resize(raw_output, (128, 128))

    elif xai_type == "LIME":
        explanation = raw_output
        top_class = explanation.top_labels[0]
        temp, mask = explanation.get_image_and_mask(top_class, positive_only=False, hide_rest=False)
        heatmap = mask.astype(np.float32)

    elif xai_type == "SHAP":
        values = raw_output.values[0]
        target_class = np.argmax(raw_output.output_names)
        heatmap = np.mean(np.abs(values[..., target_class]), axis=-1)

    # Normalização min-max [0, 1]
    if np.max(heatmap) > np.min(heatmap):
        heatmap = (heatmap - np.min(heatmap)) / (np.max(heatmap) - np.min(heatmap))
    else:
        heatmap = np.zeros_like(heatmap)

    return heatmap

# 3. Grad-CAM++

3.1 Ferramentas: Segmentação, Grad-CAM++ e lista de imagens


*   Somente rode uma vez antes do passo 3.2






In [ ]:
import numpy as np
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import os

# --- A. BANCO DE IMAGENS ---
banco_imagens = {
    "Normal": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Normal/segmented_N_109.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Normal/segmented_N_123.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Normal/segmented_N_120.jpg'
    ],
    "Glioma": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Glioma/segmented_G_121.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Glioma/segmented_G_125.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Glioma/segmented_G_131.jpg'
    ],
    "Meningioma": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Meningioma/segmented_M_11.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Meningioma/segmented_M_102.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Meningioma/segmented_M_111.jpg'
    ],
    "Pituitary": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Pituitary/segmented_P_111.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Pituitary/segmented_P_123.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Pituitary/segmented_P_1.jpg'
    ]
}


# --- C. FUNÇÃO GRAD-CAM++ (Sem Segmentação) ---
def gerar_gradcam_plus_plus(img_path, last_conv_model, classifier_model):
    # 1. Carregar a imagem original
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(128, 128))
    img_raw = tf.keras.preprocessing.image.img_to_array(img)

    # 2. Preparar para o modelo (Normalização padrão 0-1)
    # Geralmente modelos de DL esperam os pixels entre 0 e 1.
    img_normalized = img_raw
    img_batch = np.expand_dims(img_normalized, axis=0)

    # 3. Calcular gradientes
    with tf.GradientTape() as tape_1:
        with tf.GradientTape() as tape_2:
            with tf.GradientTape() as tape_3:
                conv_outputs = last_conv_model(img_batch)
                predictions = classifier_model(conv_outputs)
                best_class_index = tf.argmax(predictions[0])
                loss = predictions[:, best_class_index]

            first_grads = tape_3.gradient(loss, conv_outputs)
        second_grads = tape_2.gradient(first_grads, conv_outputs)
    third_grads = tape_1.gradient(second_grads, conv_outputs)

    # 4. Matemática do Grad-CAM++ (Igual ao anterior)
    global_sum = np.sum(conv_outputs, axis=(0, 1, 2))
    alpha_num = second_grads[0]
    alpha_denom = 2.0 * second_grads[0] + global_sum * third_grads[0]
    alpha_denom = np.where(alpha_denom != 0.0, alpha_denom, 1e-10)

    alphas = alpha_num / alpha_denom
    alphas_norm = np.sum(alphas, axis=(0,1))
    alphas_norm = np.where(alphas_norm != 0.0, alphas_norm, 1e-10)
    alphas /= alphas_norm

    weights = np.maximum(first_grads[0], 0.0)
    deep_linearization_weights = np.sum(weights * alphas, axis=(0,1))

    heatmap = np.sum(conv_outputs[0] * deep_linearization_weights, axis=-1)
    heatmap = np.maximum(heatmap, 0) # ReLU

    if np.max(heatmap) != 0:
        heatmap /= np.max(heatmap)

    # Retorna img_raw ou img_normalized para exibição
    return heatmap, img_normalized, best_class_index

print("Ferramentas carregadas - Grad-CAM++ configurado para imagens originais.")

3.2 Execução do Grad-CAM++
*  Execute uma vez por classe. Mude ``` CLASSE_ALVO ``` para o nome da classe que quer aplicar o Grad-CAM++.


In [ ]:
# ==========================================
# ESCOLHA A CLASSE AQUI
# #Normal #Glioma #Meningioma #Pituitary
# ==========================================
CLASSE_ALVO = "Pituitary"
# ==========================================

# Cria pasta para salvar
output_folder = f'/content/drive/My Drive/Projeto/GradCAM_Resultados_Para_Artigo/{CLASSE_ALVO}/'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

print(f"--- Processando Grad-CAM++ para: {CLASSE_ALVO} ---")

lista_imgs = banco_imagens[CLASSE_ALVO]

for i, img_path in enumerate(lista_imgs):
    filename = os.path.basename(img_path)
    print(f"[{i+1}/5] Gerando para: {filename}...")

    try:
        # Chama a função mestre
        heatmap, img_seg, pred_idx = gerar_gradcam_plus_plus(img_path, last_conv_model, classifier_model)

        # --- CÁLCULO DAS MÉTRICAS ---
        heatmap_2d = extract_heatmap_2d("GradCAM++", heatmap)
        score_gini = calculate_gini_index(heatmap_2d)
        score_aopc = calculate_aopc(model, img_seg, heatmap_2d, k_steps=10)

        print(f"    -> Métricas | Gini: {score_gini:.4f} | AOPC: {score_aopc:.4f}")
        # ----------------------------------

        # --- VISUALIZAÇÃO ---
        # 1. Heatmap colorido
        heatmap_resized = cv2.resize(heatmap, (128, 128))
        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

        # 2. Superposição (Usando a imagem segmentada como base)
        img_seg_uint8 = img_seg.astype(np.uint8)
        superimposed = (img_seg_uint8 * 0.6 + heatmap_colored * 0.4).astype(np.uint8)

        # 3. Plotar (3 colunas: Segmentada | Heatmap puro | Resultado)
        plt.figure(figsize=(12, 4))

        plt.subplot(1, 3, 1)
        plt.title(f"Segmentada (Entrada)")
        plt.imshow(img_seg_uint8)
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.title("Heatmap Grad-CAM++")
        plt.imshow(heatmap_resized, cmap='jet')
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.title(f"Predição: Classe {pred_idx}")
        plt.imshow(superimposed)
        plt.axis('off')

        # Salvar
        save_path = os.path.join(output_folder, f"GradCAM_{filename}")
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
        plt.close()

        print(f"    -> Salvo em: {save_path}")

    except Exception as e:
        print(f"    -> ERRO em {filename}: {e}")

print(f"\n {CLASSE_ALVO} concluído.")

# 4. LIME


4.1 Configurações iniciais (Sempre rodar assim que abrir o Colab)

In [ ]:
# --- CÉLULA DE INICIALIZAÇÃO ---

# 1. Instalar bibliotecas externas
!pip install lime

# 2. Conectar ao Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. Importar tudo o que precisa
import tensorflow as tf
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
from lime import lime_image
from skimage.segmentation import mark_boundaries

# 4. Carregar seu modelo
caminho_modelo = '/content/drive/My Drive/Projeto/PARA UPLOAD/TLSeg_128_6.keras'
try:
    model = tf.keras.models.load_model(caminho_modelo)
    print("Modelo carregado com sucesso.")
except:
    print("ERRO: Não esqueça de ajustar o caminho do modelo acima.")

# 5. Banco de imagens
banco_imagens = {
    "Normal": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Normal/segmented_N_109.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Normal/segmented_N_123.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Normal/segmented_N_120.jpg'
    ],
    "Glioma": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Glioma/segmented_G_121.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Glioma/segmented_G_125.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Glioma/segmented_G_131.jpg'
    ],
    "Meningioma": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Meningioma/segmented_M_11.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Meningioma/segmented_M_102.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Meningioma/segmented_M_111.jpg'
    ],
    "Pituitary": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Pituitary/segmented_P_111.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Pituitary/segmented_P_123.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Pituitary/segmented_P_1.jpg'
    ]
}

# 7. Preparar o LIME
explainer = lime_image.LimeImageExplainer()

def predict_fn(images):
    # O LIME manda um batch de imagens
    # O modelo já espera imagens 0-255 segmentadas
    return model.predict(images, verbose=0)

print("Sucesso - LIME configurado.")

# para Normal: N_100; N_102; N_106; N_109; N_123 -> Classe 0
# para Glioma: G_1; G_108; G_115; G_125; G_131 -> Classe 1
# para Meningioma: M_11; M_102; M_105; M_111; M_118 -> Classe 2
# para Pituitary: P_1; P_106; P_111; P_123; P_141 -> Classe 3

4.2 Visualizar o resultado

*   Opção 1: O "Comparativo" (Ideal para Slides)

**O que ela te dá:** Duas imagens coladas. A original e a explicada.

**Por que usar:** É a melhor para slides. Ela evita ter que ficar alternando imagens. Mostra o tumor original e, ao lado, prova que o LIME "circulou" exatamente aquela massa.


---
*   Opção 2: O "Prós e Contras" (Diagnóstico Profundo)

**O que ela te dá:** Uma imagem com bordas de duas cores (geralmente verde para o que confirma a classe e vermelho para o que confunde o modelo).

**Por que usar:** É excelente para análise técnica. Se o modelo errou uma predição, essa opção revela o porquê. Às vezes, o modelo vê o tumor (verde), mas vê uma sombra no osso que o faz duvidar (vermelho).


---
*   Opção 3: O "Foco Total" (Isolamento)

**O que ela te dá:** Tudo o que não é importante desaparece (fica preto). Sobram apenas os "pedaços" que o modelo considerou cruciais.

**Por que usar:** É a mais impactante visualmente. Se o modelo estiver viciado (olhando para o fundo da imagem em vez do cérebro), essa opção vai mostrar apenas um fundo preto vazio ou áreas aleatórias, "denunciando" o erro do modelo.


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from lime import lime_image
from skimage.segmentation import mark_boundaries

# ==============================================================================
# CONFIGURAÇÃO DA RODADA
# ==============================================================================
CLASSE_ALVO = "Pituitary"   # Opções: "Normal", "Glioma", "Meningioma", "Pituitary"
OPCAO_VISUAL = 3         # 1 = Comparativo | 2 = Prós/Contras | 3 = Isolamento (Fundo Preto)
# ==============================================================================

def rodar_lime_em_lote(nome_classe, opcao):

    # Validação básica
    if nome_classe not in banco_imagens:
        print(f" Erro: Classe '{nome_classe}' não existe no banco_imagens.")
        return

    # Define pasta de saída baseada na opção escolhida
    nome_opcao = f"Opcao_{opcao}"
    output_folder = f'/content/drive/My Drive/Projeto/LIME_Resultados_Para_Artigo/{nome_classe}/{nome_opcao}/'

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    lista_imgs = banco_imagens[nome_classe]
    print(f"--- Iniciando LIME (Estilo {opcao}) para {nome_classe} ---")

    for i, img_path in enumerate(lista_imgs):
        filename = os.path.basename(img_path)
        print(f"[{i+1}/5] Processando: {filename}...")

        try:
            # 1. CARREGAR E SEGMENTAR (Sempre igual)
            img_orig = tf.keras.preprocessing.image.load_img(img_path, target_size=(128, 128))
            img_raw = tf.keras.preprocessing.image.img_to_array(img_orig)
            img_segmented = img_raw
            img_lime = img_segmented.astype('double')

            # 2. RODAR O LIME
            explanation = explainer.explain_instance(
                img_lime, predict_fn, top_labels=1, hide_color=0, num_samples=1000
            )
            top_class = explanation.top_labels[0]

            # --- CÁLCULO DAS MÉTRICAS ---
            heatmap_2d = extract_heatmap_2d("LIME", explanation)
            score_gini = calculate_gini_index(heatmap_2d)
            score_aopc = calculate_aopc(model, img_segmented, heatmap_2d, k_steps=10)

            print(f"    -> Métricas | Gini: {score_gini:.4f} | AOPC: {score_aopc:.4f}")
            # ----------------------------------

            # 3. CONFIGURAR A VISUALIZAÇÃO
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

            # Título comum para a esquerda
            ax1.imshow(img_segmented.astype('uint8'))
            ax1.set_title("Entrada (Segmentada)")
            ax1.axis('off')

            # LÓGICA DAS OPÇÕES
            if opcao == 1: # LADO A LADO (Padrão)
                temp, mask = explanation.get_image_and_mask(
                    top_class, positive_only=True, num_features=5, hide_rest=False
                )
                ax2.imshow(mark_boundaries(temp / 255.0, mask))
                ax2.set_title(f"LIME: Foco Principal (Classe {top_class})")

            elif opcao == 2: # PRÓS E CONTRAS (Verde/Vermelho)
                temp, mask = explanation.get_image_and_mask(
                    top_class, positive_only=False, num_features=8, hide_rest=False
                )
                ax2.imshow(mark_boundaries(temp / 255.0, mask))
                ax2.set_title(f"Verde: Confirma | Vermelho: Contradiz")

            elif opcao == 3: # ISOLAMENTO (Fundo preto)
                temp, mask = explanation.get_image_and_mask(
                    top_class, positive_only=True, num_features=5, hide_rest=True
                )
                ax2.imshow(mark_boundaries(temp / 255.0, mask))
                ax2.set_title(f"Apenas Áreas Vitais (Isoladas)")

            ax2.axis('off')

            # 4. SALVAR
            # O nome do arquivo inclui a opção para não misturar
            nome_arquivo = f"LIME_Op{opcao}_{filename}"
            save_path = os.path.join(output_folder, nome_arquivo)

            plt.tight_layout()
            plt.savefig(save_path, bbox_inches='tight', dpi=150)
            plt.close() # Fecha para não travar a memória RAM

            print(f"    -> Salvo: {nome_arquivo}")

        except Exception as e:
            print(f"    -> Erro em {filename}: {e}")

    print(f"\n {nome_classe} (Opção {opcao}) Finalizado! Verifique a pasta no Drive.")

# --- EXECUTAR ---
rodar_lime_em_lote(CLASSE_ALVO, OPCAO_VISUAL)

# 5. SHAP

In [ ]:
import numpy as np
import tensorflow as tf
import cv2
import shap
import os
import matplotlib.pyplot as plt

# --- 1. BANCO DE IMAGENS ---
banco_imagens = {
    "Normal": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Normal/segmented_N_109.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Normal/segmented_N_123.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Normal/segmented_N_120.jpg'
    ],
    "Glioma": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Glioma/segmented_G_121.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Glioma/segmented_G_125.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Glioma/segmented_G_131.jpg'
    ],
    "Meningioma": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Meningioma/segmented_M_11.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Meningioma/segmented_M_102.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Meningioma/segmented_M_111.jpg'
    ],
    "Pituitary": [
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Pituitary/segmented_P_111.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Pituitary/segmented_P_123.jpg',
        '/content/drive/My Drive/Projeto/imagens_segmentadas/Pituitary/segmented_P_1.jpg'
    ]
}


# --- 2. FUNÇÃO DE SEGMENTAÇÃO (OpenCV) ---
def segment_image_cv(img_array_single):
    # Converte para uint8 para o OpenCV
    img = img_array_single.astype(np.uint8)
    if len(img.shape) == 4: img = np.squeeze(img, axis=0)

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    blurred = cv2.medianBlur(gray, 3)

    # 1. Otsu para máscara inicial
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 2. Detecção do Maior Contorno (evita "farelar" o cérebro)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(gray)

    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        # Preenche o contorno (máscara sólida)
        cv2.drawContours(mask, [largest_contour], -1, 255, thickness=cv2.FILLED)

        # 3. Erosão para remover o osso residual
        kernel = np.ones((3, 3), np.uint8)
        mask = cv2.erode(mask, kernel, iterations=2)

    segmented = cv2.bitwise_and(img, img, mask=mask)
    return segmented

# --- 3. PREPARAR O BACKGROUND DATA ---
background_images = []

for classe, caminhos in banco_imagens.items():
    # Pega apenas as 3 primeiras imagens de cada classe
    for caminho in caminhos[:3]:
        img = tf.keras.preprocessing.image.load_img(caminho, target_size=(128, 128))
        img_raw = tf.keras.preprocessing.image.img_to_array(img)

        img_segmented = segment_image_cv(img_raw)
        background_images.append(img_segmented)

# Converte a lista para um array NumPy (mantendo os valores de 0 a 255)
background_data = np.array(background_images)

# --- 4. CARREGAR O MODELO E CRIAR FUNÇÃO DE PREDIÇÃO ---
model_path = '/content/drive/My Drive/Projeto/PARA UPLOAD/TLSeg_128_6.keras'
model = tf.keras.models.load_model(model_path)

# O SHAP de "caixa preta" precisa apenas de uma função que receba imagens e devolva as predições
def predict_fn(imagens):
    return model.predict(imagens, verbose=0)

In [ ]:
import numpy as np
import tensorflow as tf
import cv2
import shap
import os
import matplotlib.pyplot as plt # Added matplotlib import

# --- 5. CONFIGURAÇÃO DA EXPORTAÇÃO SEM SEGMENTAÇÃO ---
# ==========================================
# ESCOLHA A CLASSE AQUI
# #Normal #Glioma #Meningioma #Pituitary
# ==========================================
CLASSE_ALVO = "Pituitary"

# Cria a pasta de destino para os resultados sem segmentação
output_dir = f'/content/drive/My Drive/Projeto/SHAP_Resultados_Para_Artigo/{CLASSE_ALVO}/'
os.makedirs(output_dir, exist_ok=True)

print(f"Iniciando geração do SHAP (Sem Segmentação) para a classe: {CLASSE_ALVO}")

# --- 6. CONFIGURAR O EXPLICADOR (Black-Box) ---
# Mantemos o shape (128, 128, 3) para compatibilidade com o modelo
masker = shap.maskers.Image("blur(128,128)", (128, 128, 3))
classes = ["Normal", "Glioma", "Meningioma", "Pituitary"]
explainer = shap.Explainer(predict_fn, masker, output_names=classes)


# --- 7. LOOP PARA PROCESSAR E SALVAR AS IMAGENS ---
for i, caminho_teste in enumerate(banco_imagens[CLASSE_ALVO]):
    nome_arquivo = os.path.basename(caminho_teste)
    print(f"[{i+1}/5] Processando {nome_arquivo}...")

    # A. Preparar a imagem: Grayscale -> RGB (128x128)
    img_teste = tf.keras.preprocessing.image.load_img(
        caminho_teste,
        target_size=(128, 128),
        color_mode='grayscale'
    )
    img_raw_teste = tf.keras.preprocessing.image.img_to_array(img_teste)

    # Converte para 3 canais para o modelo não dar erro
    img_rgb_teste = cv2.cvtColor(img_raw_teste.astype(np.uint8), cv2.COLOR_GRAY2RGB)

    # Imagem para o modelo (batch)
    img_teste_batch = np.expand_dims(img_rgb_teste.astype(np.float32), axis=0)

    # B. Calcular os valores SHAP
    shap_values = explainer(img_teste_batch, max_evals=500, batch_size=50)

    # --- CÁLCULO DAS MÉTRICAS ---
    heatmap_2d = extract_heatmap_2d("SHAP", shap_values)
    score_gini = calculate_gini_index(heatmap_2d)
    score_aopc = calculate_aopc(model, img_rgb_teste, heatmap_2d, k_steps=10)

    print(f"    -> Métricas | Gini: {score_gini:.4f} | AOPC: {score_aopc:.4f}")
    # ----------------------------------

    # C. Preparar a imagem para EXIBIÇÃO no plot (Normalizada 0-1)
    # Isso garante que a imagem real apareça nítida no primeiro quadro
    img_para_plot = np.expand_dims(img_rgb_teste.astype(np.float32) / 255.0, axis=0)

    # D. Gerar e Salvar a Imagem
    # Passamos pixel_values para forçar a imagem real no lugar da máscara
    shap.image_plot(shap_values, pixel_values=img_para_plot, show=False)

    fig = plt.gcf()
    fig.set_size_inches(18, 5)

    caminho_salvar = os.path.join(output_dir, f"SHAP_RAW_{nome_arquivo}.png")
    plt.savefig(caminho_salvar, bbox_inches='tight', dpi=300)
    plt.close()

    print(f"      Salvo em: {caminho_salvar}")

print(f"\nProcessamento da classe {CLASSE_ALVO} concluído.")